In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install menpo
!pip install fvcore
!git clone https://github.com/im-xiaoming/xiaoying.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for menpo: filename=menpo-0.11.1-py3-none-any.whl size=1611934 sha256=eb2df6f00ee6399fad04d58318dec1d776079283a790a6811dc1b23bc4839fca
  Stored in directory: /root/.cache/pip/wheels/b7/98/ad/3e7933944c49e29b1f517f0998376cc8cc4d0f53a7c6537fb6
Successfully built menpo
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=d56122ae92ce0066f017d6e6540f799a8a6d24613e0923516604257f7a033b90
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef

In [3]:
!pip install 'transformers<5'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 165.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 66.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
!cp -r /content/drive/MyDrive/Ying/data.zip /content/
!unzip /content/data.zip -d /content/
!cp -r /content/drive/MyDrive/Ying/mm.dat /content
!cp -r /content/drive/MyDrive/Ying/mm.dat.conf /content

Archive:  /content/data.zip
   creating: /content/data/
  inflating: /content/data/cfp_fp.dat  
  inflating: /content/data/lfw.dat.conf  
  inflating: /content/data/agedb_30_list.npy  
  inflating: /content/data/lfw_list.npy  
  inflating: /content/data/lfw.dat   
  inflating: /content/data/cfp_fp.dat.conf  
  inflating: /content/data/cfp_ff_list.npy  
  inflating: /content/data/agedb_30.dat.conf  
  inflating: /content/data/agedb_30.dat  
  inflating: /content/data/cfp_ff.dat  
  inflating: /content/data/cfp_ff.dat.conf  
  inflating: /content/data/cfp_fp_list.npy  


In [5]:
!cp -r /content/drive/MyDrive/Ying/faces_extracted.zip /content
!unzip -q /content/faces_extracted.zip -d /content/

!cp -r /content/drive/MyDrive/Ying/IJBB.zip /content
!unzip -q /content/IJBB.zip -d /content
!cp -r /content/drive/MyDrive/IJB_meta.zip /content
!unzip -q /content/IJB_meta.zip -d /content
!cp -r /content/meta/IJBB_meta/* /content/IJBB/meta

In [ ]:
# !cp -r /content/drive/MyDrive/ijb-testsuite.tar /content
# !file ijb-testsuite.tar
# !tar -xf ijb-testsuite.tar
# !cp -r /content/drive/MyDrive/IJB_meta.zip /content
# !unzip -q /content/IJB_meta.zip -d /content
# !cp -r /content/meta/IJBB_meta/* /content/ijb/IJBB/meta
# !cp -r /content/meta/IJBC_meta/* /content/ijb/IJBC/meta

In [6]:
import numpy as np
from tqdm import tqdm
import os
import json
from PIL import Image
import cv2

import torch
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import DataLoader

from xiaoying.validation import evaluate_utils, evaluate
from xiaoying import net
from xiaoying.data import CustomImageFolderDataset, val_dataset
from xiaoying.head import AdaFace, SubAdaFace
from xiaoying.utils import EarlyStopping, load_checkpoint, load_weights
from xiaoying.models import get_model
from xiaoying.ViT.load_models import load_vit_base

In [47]:
root = '/content/faces_extracted'
print("Num claass: ", len(os.listdir(root)))

train_transforms = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

train_dataset = CustomImageFolderDataset('/content/faces_extracted',
                                         transform=train_transforms)
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)


val_ds = val_dataset(data_root='/content/data')
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False,
                        num_workers=8, pin_memory=True)

Num claass:  8631
loading validation data memfile
loading validation data memfile
loading validation data memfile
loading validation data memfile


In [8]:
def split_parameters(model):
    decay = []
    no_decay = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if (
            len(param.shape) == 1
            or name.endswith(".bias")
            or "pos_embed" in name
        ):
            no_decay.append(param)
        else:
            decay.append(param)

    return decay, no_decay

In [23]:
model_name = 'vit_base'
model = load_vit_base()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

criterion = nn.CrossEntropyLoss()
head = AdaFace(embedding_size=512, classnum=8631, m=0.4, h=0.333, s=64., t_alpha=0.99)
head.to(device)

paras_wo_bn, paras_only_bn = split_parameters(model)

optimizer = torch.optim.SGD([{
            'params': paras_wo_bn + [head.kernel],
            'weight_decay': 1e-4
        }, {
            'params': paras_only_bn
        }], lr=0.001, momentum=0.9)

Loaded ViT model
compatible keys in state_dict 281 / 281
Check


<All keys matched successfully>
Loaded pretrained model from pretrained_model/model.pt
Total parameters: 114,869,760
Trainable parameters: 114,869,760
AdaFace with the following property
self.m 0.4
self.h 0.333
self.s 64.0
self.t_alpha 0.99


In [10]:
def validate(model, dataname, val_loader, device):
    # Evaluate
    print('Evaluate step 1...')
    mean_acc = evaluate.evaluate1(model, val_loader, device)

    print('Evaluate step 2...')
    r = evaluate.evaluate2('', model, 'IR', 'IJBB', 512, device=device)

    return mean_acc, r

In [11]:
from torch.cuda.amp import autocast, GradScaler

In [20]:
def train(model, head, optimizer, train_loader,
          criterion, epochs, device, start):

    scaler = GradScaler()
    for epoch in range(start, epochs):
        # ==================== Training ==================== #
        model.train()

        train_loss = []
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        print('\nTrain...')
        for images, labels in pbar:

            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # forward
            with autocast():
                embedings = model(images)
                norm = torch.norm(embedings, p=2, dim=-1, keepdim=True)
                embedings = torch.div(embedings, norm)

                cos_theta = head(embedings, norm, labels)
                loss = criterion(cos_theta, labels)

            # backward
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss.append(loss.item())

            # update tqdm
            pbar.set_postfix({
                "loss": f"{np.mean(train_loss):.4f}",
                "lr": optimizer.param_groups[0]["lr"]
            })

In [24]:
state_dict = torch.load('/content/drive/MyDrive/temp_checkpoint.pth')
model.load_state_dict(state_dict['model_state_dict'])
head.load_state_dict(state_dict['head_state_dict'])
optimizer.load_state_dict(state_dict['optimizer_state_dict'])

In [45]:
for param_group in optimizer.param_groups:
    param_group['lr'] = 0.000001

print(optimizer.param_groups[0]['lr'])

1e-06


In [48]:
train(model, head, optimizer, train_loader, criterion, epochs=3, device=device, start=2)

Epoch 3/3:   0%|          | 0/12258 [00:00<?, ?it/s]


Train...


Epoch 3/3:   4%|▍         | 541/12258 [03:28<1:15:06,  2.60it/s, loss=2.9662, lr=1e-6]


KeyboardInterrupt: 

In [ ]:
import gc
gc.collect()

454

In [49]:
torch.save(model.state_dict(), 'ViT_base.pth')

In [52]:
!cp -r /content/ViT_base.pth /content/drive/MyDrive